## **Loan Portfolio Risk & Performance Analytics — LendingClub**

Loan Portfolio Risk & Performance Analytics — LendingClub Case Study
Background:
LendingClub is a peer-to-peer (P2P) lending platform that connects borrowers seeking personal loans with investors willing to fund them. Between 2007 and 2018, LendingClub issued over 2.2 million loans totalling billions of dollars across the United States — making it the world's largest P2P lending platform at its peak. Unlike traditional banks, LendingClub operates without branches, relying entirely on data-driven credit assessment to approve loans, assign interest rates, and manage risk.

Business Problem:
In a lending business, every loan that defaults represents a direct financial loss — the principal is written off as a charge-off and reduces the company's profitability. LendingClub's core challenge is: how do you lend to as many borrowers as possible (to grow revenue) while keeping defaults low enough (to protect margins)? This tension between growth and risk sits at the heart of every credit business.

As a Data Analyst, the business expects answers to four critical questions:
1. Which borrower segments carry the highest default risk, and what patterns predict default?
2. Is the loan portfolio growing, and which loan grades, purposes, and geographies drive that growth?
3. Are loans priced correctly — does the interest rate assigned to each grade cover the actual default loss?
4. How much credit loss (charge-offs) is the business absorbing, and what is being recovered?

Objective:
This project builds an end-to-end analytics solution — from raw data to interactive dashboard — that gives LendingClub's credit, pricing, and finance teams a single source of truth to monitor portfolio health, identify high-risk borrower profiles, track origination trends, and measure credit loss. The goal is not just to visualise data but to produce actionable insights that directly support lending decisions.

Dataset:
The dataset contains 890,000+ real loan records sourced from LendingClub's SEC filings (2007–2018), with 74+ columns covering borrower financials (income, DTI, FICO score, credit utilisation), loan attributes (grade, interest rate, purpose, term), and repayment outcomes (loan status, charge-off amount, recoveries).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

## set visual style for reporting
sns.set_theme(style='whitegrid')
warnings.filterwarnings('ignore')


In [2]:
df1 = pd.read_csv('accepted_2007_to_2018Q4.csv')


In [3]:
df1.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df1.shape

(2260701, 151)

In [5]:
df1.columns

Index(['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv',
       'term', 'int_rate', 'installment', 'grade', 'sub_grade',
       ...
       'hardship_payoff_balance_amount', 'hardship_last_payment_amount',
       'disbursement_method', 'debt_settlement_flag',
       'debt_settlement_flag_date', 'settlement_status', 'settlement_date',
       'settlement_amount', 'settlement_percentage', 'settlement_term'],
      dtype='str', length=151)

In [6]:
df1['emp_length'].unique()

<ArrowStringArray>
['10+ years',   '3 years',   '4 years',   '6 years',    '1 year',   '7 years',
   '8 years',   '5 years',   '2 years',   '9 years',  '< 1 year',         nan]
Length: 12, dtype: str

In [7]:
df1['home_ownership'].unique()

<ArrowStringArray>
['MORTGAGE', 'RENT', 'OWN', 'ANY', nan, 'NONE', 'OTHER']
Length: 7, dtype: str

In [8]:
df1['annual_inc']

0           55000.0
1           65000.0
2           63000.0
3          110000.0
4          104433.0
             ...   
2260696    227000.0
2260697    110000.0
2260698     95000.0
2260699         NaN
2260700         NaN
Name: annual_inc, Length: 2260701, dtype: float64

In [9]:
df1['verification_status'].unique()

<ArrowStringArray>
['Not Verified', 'Source Verified', 'Verified', nan]
Length: 4, dtype: str

In [10]:
df1['issue_d']

0          Dec-2015
1          Dec-2015
2          Dec-2015
3          Dec-2015
4          Dec-2015
             ...   
2260696    Oct-2016
2260697    Oct-2016
2260698    Oct-2016
2260699         NaN
2260700         NaN
Name: issue_d, Length: 2260701, dtype: str

In [11]:
df1['loan_status'].unique()

<ArrowStringArray>
[                                         'Fully Paid',
                                             'Current',
                                         'Charged Off',
                                     'In Grace Period',
                                  'Late (31-120 days)',
                                   'Late (16-30 days)',
                                             'Default',
                                                   nan,
  'Does not meet the credit policy. Status:Fully Paid',
 'Does not meet the credit policy. Status:Charged Off']
Length: 10, dtype: str

In [12]:
df1['pymnt_plan'].unique()

<ArrowStringArray>
['n', 'y', nan]
Length: 3, dtype: str

In [13]:
df1['purpose'].unique()

<ArrowStringArray>
['debt_consolidation',     'small_business',   'home_improvement',
     'major_purchase',        'credit_card',              'other',
              'house',           'vacation',                'car',
            'medical',             'moving',   'renewable_energy',
            'wedding',        'educational',                  nan]
Length: 15, dtype: str

In [14]:
df1['dti']

0           5.91
1          16.06
2          10.78
3          17.06
4          25.37
           ...  
2260696    12.75
2260697    18.30
2260698    23.36
2260699      NaN
2260700      NaN
Name: dti, Length: 2260701, dtype: float64

In [15]:
df1['fico_range_high']

0          679.0
1          719.0
2          699.0
3          789.0
4          699.0
           ...  
2260696    709.0
2260697    664.0
2260698    664.0
2260699      NaN
2260700      NaN
Name: fico_range_high, Length: 2260701, dtype: float64

In [16]:
df1['fico_range_low']

0          675.0
1          715.0
2          695.0
3          785.0
4          695.0
           ...  
2260696    705.0
2260697    660.0
2260698    660.0
2260699      NaN
2260700      NaN
Name: fico_range_low, Length: 2260701, dtype: float64

In [17]:
df1['last_fico_range_high']

0          564.0
1          699.0
2          704.0
3          679.0
4          704.0
           ...  
2260696    724.0
2260697    594.0
2260698    669.0
2260699      NaN
2260700      NaN
Name: last_fico_range_high, Length: 2260701, dtype: float64

In [18]:
df1['last_fico_range_low']

0          560.0
1          695.0
2          700.0
3          675.0
4          700.0
           ...  
2260696    720.0
2260697    590.0
2260698    665.0
2260699      NaN
2260700      NaN
Name: last_fico_range_low, Length: 2260701, dtype: float64

In [19]:
df1['earliest_cr_line']

0          Aug-2003
1          Dec-1999
2          Aug-2000
3          Sep-2008
4          Jun-1998
             ...   
2260696    Feb-1995
2260697    Jul-1999
2260698    Jun-1996
2260699         NaN
2260700         NaN
Name: earliest_cr_line, Length: 2260701, dtype: str

In [20]:
df1['open_acc']

0           7.0
1          22.0
2           6.0
3          13.0
4          12.0
           ... 
2260696     5.0
2260697    10.0
2260698     8.0
2260699     NaN
2260700     NaN
Name: open_acc, Length: 2260701, dtype: float64

In [21]:
df1['pub_rec']

0          0.0
1          0.0
2          0.0
3          0.0
4          0.0
          ... 
2260696    0.0
2260697    1.0
2260698    0.0
2260699    NaN
2260700    NaN
Name: pub_rec, Length: 2260701, dtype: float64

In [22]:
df1['revol_bal']

0           2765.0
1          21470.0
2           7869.0
3           7802.0
4          21929.0
            ...   
2260696     8633.0
2260697    17641.0
2260698     7662.0
2260699        NaN
2260700        NaN
Name: revol_bal, Length: 2260701, dtype: float64

In [23]:
df1['application_type'].unique()

<ArrowStringArray>
['Individual', 'Joint App', nan]
Length: 3, dtype: str

In [24]:
df1['total_pymnt']

0           4421.723917
1          25679.660000
2          22705.924294
3          31464.010000
4          11740.500000
               ...     
2260696    24903.930000
2260697     6755.400000
2260698     9621.250000
2260699             NaN
2260700             NaN
Name: total_pymnt, Length: 2260701, dtype: float64

In [25]:
df1.columns

Index(['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv',
       'term', 'int_rate', 'installment', 'grade', 'sub_grade',
       ...
       'hardship_payoff_balance_amount', 'hardship_last_payment_amount',
       'disbursement_method', 'debt_settlement_flag',
       'debt_settlement_flag_date', 'settlement_status', 'settlement_date',
       'settlement_amount', 'settlement_percentage', 'settlement_term'],
      dtype='str', length=151)

In [26]:
features = ['id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 
            'grade', 'sub_grade', 'emp_length', 'home_ownership', 'annual_inc',
            'verification_status', 'issue_d', 'loan_status', 'purpose',
            'fico_range_high', 'fico_range_low', 'dti', 
            'earliest_cr_line',  'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
            'application_type', 'mort_acc', 'pub_rec_bankruptcies' ]

In [27]:
new_df = df1[features]

In [28]:
new_df.head()

,id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,...,dti,earliest_cr_line,open_acc,pub_rec,revol_bal,revol_util,total_acc,application_type,mort_acc,pub_rec_bankruptcies
0,68407277,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,10+ years,...,5.91,Aug-2003,7.0,0.0,2765.0,29.7,13.0,Individual,1.0,0.0
1,68355089,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,10+ years,...,16.06,Dec-1999,22.0,0.0,21470.0,19.2,38.0,Individual,4.0,0.0
2,68341763,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,10+ years,...,10.78,Aug-2000,6.0,0.0,7869.0,56.2,18.0,Joint App,5.0,0.0
3,66310712,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,10+ years,...,17.06,Sep-2008,13.0,0.0,7802.0,11.6,17.0,Individual,1.0,0.0
4,68476807,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,3 years,...,25.37,Jun-1998,12.0,0.0,21929.0,64.5,35.0,Individual,6.0,0.0


In [29]:
new_df.shape

(2260701, 28)

In [30]:
new_df['loan_status'].unique()

<ArrowStringArray>
[                                         'Fully Paid',
                                             'Current',
                                         'Charged Off',
                                     'In Grace Period',
                                  'Late (31-120 days)',
                                   'Late (16-30 days)',
                                             'Default',
                                                   nan,
  'Does not meet the credit policy. Status:Fully Paid',
 'Does not meet the credit policy. Status:Charged Off']
Length: 10, dtype: str

In [31]:
target_status = ['Fully Paid', 'Charged Off']
new_df = new_df[new_df['loan_status'].isin(target_status)]

In [32]:
new_df.shape

(1345310, 28)

In [33]:
new_df.isnull().sum()

id                          0
loan_amnt                   0
funded_amnt                 0
funded_amnt_inv             0
term                        0
int_rate                    0
installment                 0
grade                       0
sub_grade                   0
emp_length              78511
home_ownership              0
annual_inc                  0
verification_status         0
issue_d                     0
loan_status                 0
purpose                     0
fico_range_high             0
fico_range_low              0
dti                       374
earliest_cr_line            0
open_acc                    0
pub_rec                     0
revol_bal                   0
revol_util                857
total_acc                   0
application_type            0
mort_acc                47281
pub_rec_bankruptcies      697
dtype: int64

In [34]:
critical_cols = ['emp_length', 'dti', 'revol_util', 'annual_inc']

new_df.dropna(subset=critical_cols, inplace=True)

In [35]:
new_df.shape

(1265976, 28)

In [36]:
new_df.isnull().sum()

id                          0
loan_amnt                   0
funded_amnt                 0
funded_amnt_inv             0
term                        0
int_rate                    0
installment                 0
grade                       0
sub_grade                   0
emp_length                  0
home_ownership              0
annual_inc                  0
verification_status         0
issue_d                     0
loan_status                 0
purpose                     0
fico_range_high             0
fico_range_low              0
dti                         0
earliest_cr_line            0
open_acc                    0
pub_rec                     0
revol_bal                   0
revol_util                  0
total_acc                   0
application_type            0
mort_acc                45884
pub_rec_bankruptcies      697
dtype: int64

In [37]:
new_df['avg_fico'] = (new_df['fico_range_high'] + new_df['fico_range_low'])/2

In [38]:
new_df.describe()

,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,annual_inc,fico_range_high,fico_range_low,dti,open_acc,pub_rec,revol_bal,revol_util,total_acc,mort_acc,pub_rec_bankruptcies,avg_fico
count,1.265976e+06,1.265976e+06,1.265976e+06,1.265976e+06,1.265976e+06,1.265976e+06,1.265976e+06,1.265976e+06,1.265976e+06,1.265976e+06,1.265976e+06,1.265976e+06,1.265976e+06,1.265976e+06,1.220092e+06,1.265279e+06,1.265976e+06
mean,1.460269e+04,1.459391e+04,1.457060e+04,1.323256e+01,4.429736e+02,7.788702e+04,7.001179e+02,6.961178e+02,1.813085e+01,1.167670e+01,2.090474e-01,1.644234e+04,5.205292e+01,2.507274e+01,1.662856e+00,1.294916e-01,6.981178e+02
std,8.745416e+03,8.741509e+03,8.744528e+03,4.769247e+00,2.624108e+02,7.101853e+04,3.165790e+01,3.165730e+01,9.569786e+00,5.488948e+00,5.976770e-01,2.253838e+04,2.448106e+01,1.201031e+01,1.996019e+00,3.723424e-01,3.165760e+01
min,5.000000e+02,5.000000e+02,0.000000e+00,5.310000e+00,4.930000e+00,3.300000e+01,6.290000e+02,6.250000e+02,-1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.000000e+00,0.000000e+00,0.000000e+00,6.270000e+02
25%,8.000000e+03,8.000000e+03,8.000000e+03,9.750000e+00,2.521400e+02,4.800000e+04,6.740000e+02,6.700000e+02,1.176000e+01,8.000000e+00,0.000000e+00,6.046000e+03,3.380000e+01,1.600000e+01,0.000000e+00,0.000000e+00,6.720000e+02
50%,1.207500e+04,1.205000e+04,1.200000e+04,1.274000e+01,3.794500e+02,6.500000e+04,6.940000e+02,6.900000e+02,1.752000e+01,1.100000e+01,0.000000e+00,1.128800e+04,5.250000e+01,2.300000e+01,1.000000e+00,0.000000e+00,6.920000e+02
75%,2.000000e+04,2.000000e+04,2.000000e+04,1.599000e+01,5.873400e+02,9.250000e+04,7.140000e+02,7.100000e+02,2.391000e+01,1.400000e+01,0.000000e+00,1.998000e+04,7.100000e+01,3.200000e+01,3.000000e+00,0.000000e+00,7.120000e+02
max,4.000000e+04,4.000000e+04,4.000000e+04,3.099000e+01,1.719830e+03,1.099920e+07,8.500000e+02,8.450000e+02,9.990000e+02,9.000000e+01,8.600000e+01,2.904836e+06,8.923000e+02,1.760000e+02,5.100000e+01,1.200000e+01,8.475000e+02


In [39]:
new_df.info()

<class 'pandas.DataFrame'>
Index: 1265976 entries, 0 to 2260697
Data columns (total 29 columns):
 #   Column                Non-Null Count    Dtype  
---  ------                --------------    -----  
 0   id                    1265976 non-null  object 
 1   loan_amnt             1265976 non-null  float64
 2   funded_amnt           1265976 non-null  float64
 3   funded_amnt_inv       1265976 non-null  float64
 4   term                  1265976 non-null  str    
 5   int_rate              1265976 non-null  float64
 6   installment           1265976 non-null  float64
 7   grade                 1265976 non-null  str    
 8   sub_grade             1265976 non-null  str    
 9   emp_length            1265976 non-null  str    
 10  home_ownership        1265976 non-null  str    
 11  annual_inc            1265976 non-null  float64
 12  verification_status   1265976 non-null  str    
 13  issue_d               1265976 non-null  str    
 14  loan_status           1265976 non-null  str    
 1

In [40]:
new_df['issue_d'] = pd.to_datetime(new_df['issue_d'])

In [41]:
new_df['earliest_cr_line'] = pd.to_datetime(new_df['issue_d'])

In [42]:
new_df.drop(columns=['fico_range_high', 'fico_range_low'], inplace=True)

In [43]:
new_df['default_flag'] = np.where(new_df['loan_status'] == 'Charged Off', 1, 0)

In [44]:
new_df.shape

(1265976, 28)

In [45]:
la = sum(new_df['loan_amnt'])

In [46]:
fa = sum(new_df['funded_amnt'])

In [47]:
fai = sum(new_df['funded_amnt_inv'])

In [48]:
la - fa

11116875.0

In [49]:
fa - fai

29503793.604370117

In [50]:
new_df.info()

<class 'pandas.DataFrame'>
Index: 1265976 entries, 0 to 2260697
Data columns (total 28 columns):
 #   Column                Non-Null Count    Dtype         
---  ------                --------------    -----         
 0   id                    1265976 non-null  object        
 1   loan_amnt             1265976 non-null  float64       
 2   funded_amnt           1265976 non-null  float64       
 3   funded_amnt_inv       1265976 non-null  float64       
 4   term                  1265976 non-null  str           
 5   int_rate              1265976 non-null  float64       
 6   installment           1265976 non-null  float64       
 7   grade                 1265976 non-null  str           
 8   sub_grade             1265976 non-null  str           
 9   emp_length            1265976 non-null  str           
 10  home_ownership        1265976 non-null  str           
 11  annual_inc            1265976 non-null  float64       
 12  verification_status   1265976 non-null  str           
 13

In [51]:
min(new_df['issue_d'])

Timestamp('2007-06-01 00:00:00')

In [52]:
max(new_df['issue_d'])

Timestamp('2018-12-01 00:00:00')